# CNN Model
Train and evaluate deep learning model on imaging data.


In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

# Setup paths for local VS Code environment
BASE_PATH = r'C:\Users\User\Desktop\Lung Cancer Detection\data\processed'
CSV_PATH = os.path.join(BASE_PATH, 'labels.csv')
PATCH_PATH = os.path.join(BASE_PATH, 'patches')

# Load labels
df = pd.read_csv(CSV_PATH)

# Split into train and validation
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

print(f"Total patches: {len(df)}")
print(f"Training on: {len(train_df)}")
print(f"Validating on: {len(val_df)}")


In [ ]:
import os
import numpy as np
import pandas as pd
import gc

def data_generator(dataframe, batch_size=4, patch_size=64):
    patches_processed = 0
    while True:
        df_shuffled = dataframe.sample(frac=1).reset_index(drop=True)
        yielded_in_epoch = False

        for i in range(0, len(df_shuffled), batch_size):
            batch_df = df_shuffled.iloc[i:i + batch_size]
            X, y = [], []

            for _, row in batch_df.iterrows():
                path = os.path.join(PATCH_PATH, row["patch_file"])
                if not os.path.exists(path):
                    continue

                try:
                    patch = np.load(path, mmap_mode='r')
                except Exception:
                    continue

                if patch.shape != (patch_size, patch_size, patch_size):
                    continue

                X.append(np.array(patch, dtype=np.float32))
                y.append(int(row["label"]))
                patches_processed += 1

            if len(X) == 0:
                continue

            X_batch = np.expand_dims(np.stack(X, axis=0), axis=-1)
            y_batch = np.array(y, dtype=np.int32)

            yielded_in_epoch = True
            yield X_batch, y_batch

            del X_batch, y_batch, X, y
            if patches_processed >= 10:
                gc.collect()
                patches_processed = 0

        if not yielded_in_epoch:
            raise RuntimeError("No valid patches found.")

train_gen = data_generator(train_df, batch_size=4)
val_gen = data_generator(val_df, batch_size=4)
print(" Generators ready!")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(64, 64, 64, 1)),

    layers.Conv3D(32, kernel_size=3, activation="relu", padding="same"),
    layers.MaxPooling3D(pool_size=2),

    layers.Conv3D(64, kernel_size=3, activation="relu", padding="same"),
    layers.MaxPooling3D(pool_size=2),

    layers.Conv3D(128, kernel_size=3, activation="relu", padding="same"),
    layers.MaxPooling3D(pool_size=2),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print(" 3D CNN Model built!")
model.summary()

In [ ]:
# 1. Start the training process
print("Starting training...")
history = model.fit(
    train_gen,
    steps_per_epoch=len(train_df) // 4,  # dividing by batch_size (4)
    validation_data=val_gen,
    validation_steps=len(val_df) // 4,
    epochs=10  # It will pass through the data 10 times
)

# 2. Save the trained model to your Google Drive!
save_path = '/content/drive/MyDrive/lung_cancer_3d_cnn.h5'
model.save(save_path)
print(f" Model successfully saved to {save_path}")

In [ ]:
import matplotlib.pyplot as plt

# Plot accuracy
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import os

# 1. Load your pre-trained Colab model
model_path = r'C:\Users\User\Desktop\Lung Cancer Detection\models\lung_cancer_3d_cnn.h5'

if os.path.exists(model_path):
    trained_model = load_model(model_path)
    print("✅ Colab model loaded successfully!")

    # 2. Test it on a real patch
    sample_x, sample_y = next(val_gen)
    test_patch = sample_x[0:1] 
    actual_label = sample_y[0]

    prediction = trained_model.predict(test_patch)[0][0]

    print(f"Actual Label:   {'Malignant (1)' if actual_label == 1 else 'Benign (0)'}")
    print(f"Model Truth %:  {prediction * 100:.2f}% chance of being Malignant")

    if prediction > 0.5:
        print("Model Decision: 🚨 It thinks this is Malignant Cancer")
    else:
        print("Model Decision: ✅ It thinks this is Benign")
else:
    print(f"❌ Model file not found at: {model_path}\nPlease make sure the .h5 file is in the models folder.")

SyntaxError: unmatched '}' (3651907678.py, line 28)